In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "demo")))

# Optimus Demo

## Sample Loading

In [ ]:
from IPython.display import clear_output
import pandas as pd
df = pd.read_pickle('demo/data/demo_sample.pkl')
df.head(5)

In [ ]:
FT_SPEC = {
    "ip_geo_ip_static_ip_score": 'optimal',
    "ip_geo_ip_connection_type": False,
    "ip_geo_ip_user_type": False,
    "ip_geo_ip_is_anonymous": False,
    "ip_geo_ip_is_anonymous_proxy": False,
    "ip_geo_ip_is_anonymous_vpn": False,
    "ip_geo_ip_is_hosting_provider": False,
    "ip_geo_ip_is_legitimate_proxy": False,
    "ip_geo_ip_is_public_proxy": False,
    "ip_geo_ip_is_residential_proxy": False,
    "ip_geo_ip_is_tor_exit_node": False,
    "bw_device_type": False,
    "bw_device_browser": False,
    "bw_device_screen_size": 'optimal',
}
LABEL = "is_failed_kyc"
MISSING_VALUES = ["__N.A__", "__C.N.A__", -990000, -999998, -999999]
FS_PARAMS = {
    'corr_threshold': 0.98,
    'psi_threshold': 0.1,
    'iv_threshold': 0.01,
    'vif_threshold': 10,
    'boosting_select_frac': 1,
    'stability_threshold': 0.05,
}

## Training

In [ ]:
from optimus.trainer import Train
trainer = Train(
    model_path="demo/models",
    report_path="demo/reports",
    model_type="LR",
    missing_values=MISSING_VALUES,
    tune_method="BO",
    score_floor=0,
    score_cap=1,
    n_bins=10,
    max_evals=10,
    calibration_method="isotonic",
    score_bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    version="4.0.0",
    spec=FT_SPEC,
    ignore_preprocessors=['VIF', 'Boosting', 'Stability'],
    **FS_PARAMS,
)
X = df[FT_SPEC.keys()]
y = df[LABEL]
e = df[df.columns.difference(FT_SPEC.keys()).tolist()]
performance = trainer.fit(X, y, e).transform(X, y, e)
clear_output()
trainer.write_report()

## Transformation only

In [ ]:
from optimus.trainer import Train

TS = '20260204_134531'
trainer = Train('demo/models')
X = df[FT_SPEC.keys()]
y = df[LABEL]
e = df[df.columns.difference(FT_SPEC.keys()).tolist()]
performance = trainer.transform(X, y, e, ts=TS)
clear_output()

## Refitting

In [ ]:
TS = '20260204_134531'
trainer.refit_model(ts=TS, trial_index=5)
performance = trainer.transform(X, y, e, ts=TS)
clear_output()

## Quick EDA

In [ ]:
from optimus.encoder import Encoder

X = df[FT_SPEC.keys()]
y = df[LABEL]
encoder = Encoder(spec=FT_SPEC, missing_values=MISSING_VALUES).fit(X, y)
clear_output()

In [ ]:
woe_df = encoder.get_woe_df(X, y)
ft_summary = woe_df['summary']
ft_bin = woe_df['binning']

In [ ]:
from eda_helper import FeatureBrowser

fb = FeatureBrowser(ft_bin)
fb.display()